# Data Cleaning with Pandas

Cleaning real-world data in 7 steps:

1. Missing values
2. Check missing values
3. Duplicates
4. Data types
5. Rename columns
6. Text cleaning (string operations)
7. Cleaning pipeline (recommended order)

> Remember: Clean data → Reliable analysis → Better insights

## Create a messy dataset
This data has missing values, a duplicate row, numbers stored as text, dates stored as text, bad column names and messy text.

In [ ]:
import pandas as pd
import numpy as np

raw = {
    'ID':      [1, 2, 3, 3, 4, 5, 6],
    'old_name':[' Alice ', 'BOB', 'Cha rlie', 'Cha rlie', None, 'Divya', 'kiran'],
    'old_age': ['25', '30', '28', '28', None, '35', 'abc'],
    'city':    ['Delhi', 'Mumbai', 'Chennai', 'Chennai', 'Delhi', None, 'Pune'],
    'salary':  [50000, None, 45000, 45000, 60000, None, None],
    'date':    ['2024-01-01', '2024-01-02', '2024-01-03', '2024-01-03', '2024-01-04', '2024-01-05', '2024-01-06']
}

df = pd.DataFrame(raw)
df

In [ ]:
df.info()

---
## 1. Missing Values
- `isna()` / `isnull()` → detect missing values
- `dropna()` → drop rows/columns with missing values
- `fillna(0)` → fill missing values with a specified value

In [ ]:
# Detect missing values (True = missing)
df.isna()

In [ ]:
# isnull() does the same thing as isna()
df.isnull().head(3)

---
## 2. Check Missing Values
`isna().sum()` returns the count of missing values for each column.

In [ ]:
df.isna().sum()

---
## 3. Duplicates
- `duplicated()` → detect duplicate rows
- `drop_duplicates()` → remove duplicate rows

In [ ]:
# True = this row is a duplicate of an earlier row
df.duplicated()

In [ ]:
# Show the duplicate rows
df[df.duplicated()]

In [ ]:
# Before
print('Rows before:', len(df))

df = df.drop_duplicates()

# After
print('Rows after :', len(df))
df

---
## 4. Data Types
- `astype(int)` → change data type
- `pd.to_numeric()` → convert to numeric
- `pd.to_datetime()` → convert to datetime

In [ ]:
# Before: age and date are 'object' (text)
df.dtypes

In [ ]:
# errors='coerce' turns bad values like 'abc' into NaN
df['old_age'] = pd.to_numeric(df['old_age'], errors='coerce')
df['date']    = pd.to_datetime(df['date'])

# After
df.dtypes

---
## 5. Rename Columns
`df.rename(columns={"old": "new"})` renames one or multiple columns.

In [ ]:
df = df.rename(columns={'old_name': 'Name', 'old_age': 'Age'})
df.columns.tolist()

---
## 6. Text Cleaning (String Operations)
- `str.strip()` → remove extra spaces
- `str.lower()` → lower case
- `str.upper()` → upper case
- `str.replace(" ", "_")` → replace text
- `str.contains("A")` → check if a pattern exists

> **TIP:** String functions help clean text before analysis.

In [ ]:
print('Before:')
print(df['Name'].tolist())

df['Name'] = df['Name'].str.strip()      # remove extra spaces
print('\nAfter strip():')
print(df['Name'].tolist())

In [ ]:
print('lower():', df['Name'].str.lower().tolist())
print('upper():', df['Name'].str.upper().tolist())
print('replace():', df['Name'].str.replace(' ', '_').tolist())

In [ ]:
# Check if the pattern exists (case-insensitive, skips missing)
df['Name'].str.contains('a', case=False, na=False)

In [ ]:
# Apply: replace space with _, then make lower case
df['Name'] = df['Name'].str.replace(' ', '_').str.lower()
df[['ID', 'Name']]

---
## 7. Cleaning Pipeline (Recommended Order)

**Missing Values → Duplicates → Data Types → Text Cleaning → Validation**

Let's run the full pipeline on a fresh copy of the raw data.

In [ ]:
# Fresh messy data
df2 = pd.DataFrame(raw)
print('Start shape:', df2.shape)

# Step 1: Missing values - detect and handle
print('\nMissing values:')
print(df2.isna().sum())
df2['city']   = df2['city'].fillna('Unknown')
df2['salary'] = df2['salary'].fillna(df2['salary'].median())
df2 = df2.dropna(subset=['old_name'])           # drop rows with no name

# Step 2: Duplicates - remove duplicate records
df2 = df2.drop_duplicates()

# Step 3: Data types - convert to correct types
df2['old_age'] = pd.to_numeric(df2['old_age'], errors='coerce')
df2['old_age'] = df2['old_age'].fillna(df2['old_age'].median()).astype(int)
df2['date']    = pd.to_datetime(df2['date'])

# Rename columns
df2 = df2.rename(columns={'old_name': 'Name', 'old_age': 'Age'})

# Step 4: Text cleaning - clean and standardize text
df2['Name'] = df2['Name'].str.strip().str.replace(' ', '_').str.lower()
df2['city'] = df2['city'].str.strip().str.title()

df2

### Validation
Check the final data quality.

In [ ]:
print('Missing values left:', df2.isna().sum().sum())
print('Duplicate rows left :', df2.duplicated().sum())
print()
print(df2.dtypes)

In [ ]:
# Simple automatic checks
assert df2.isna().sum().sum() == 0, 'Still has missing values!'
assert df2.duplicated().sum() == 0, 'Still has duplicates!'
assert df2['Age'].dtype == 'int64', 'Age should be int'
print('✅ All validation checks passed')

## Summary
| Step | Functions |
|---|---|
| 1. Missing values | `isna()`, `isnull()`, `dropna()`, `fillna()` |
| 2. Check missing | `isna().sum()` |
| 3. Duplicates | `duplicated()`, `drop_duplicates()` |
| 4. Data types | `astype()`, `to_numeric()`, `to_datetime()` |
| 5. Rename columns | `rename(columns={...})` |
| 6. Text cleaning | `str.strip()`, `lower()`, `upper()`, `replace()`, `contains()` |
| 7. Pipeline | Missing → Duplicates → Types → Text → Validation |

**Garbage in, garbage out. Always clean your data before analysis!**